# Part 2: ETL & Data Preparation (Data Engineer)

## 🎯 Learning Objectives
- Understand ETL (Extract, Transform, Load) process
- Clean datasets **separately** before merging (real-world best practice)
- Implement Bronze → Silver → Gold data layers
- Prepare clean, analysis-ready data

## 📊 Data Architecture (Medallion Pattern)

```
BRONZE (Raw)          SILVER (Cleaned)         GOLD (Business-Ready)
├── reviews.csv   →   ├── clean_reviews.csv   →  final_dataset.csv
├── inventory.csv →   ├── clean_inventory.csv →  (merged + enriched)
└── sales.csv     →   └── clean_sales.csv     →  
```

**Why separate cleaning?**
- Scales better with large datasets
- Each team can own their data source
- Easier to debug issues
- Industry standard (Databricks, Azure, AWS)

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

print("✅ Libraries loaded")
print("\n📍 Current Phase: BRONZE → SILVER (Cleaning)")

## Step 1: Load Bronze (Raw) Data

In [ ]:
# Load raw data from Bronze layer
reviews_raw = pd.read_csv('../data/raw/reviews.csv')
inventory_raw = pd.read_csv('../data/raw/inventory.csv')
sales_raw = pd.read_csv('../data/raw/sales.csv')

print("📦 Bronze Layer Loaded:")
print(f"  Reviews: {len(reviews_raw):,} rows")
print(f"  Inventory: {len(inventory_raw):,} rows")
print(f"  Sales: {len(sales_raw):,} rows")

## Step 2: Clean Reviews Dataset (Bronze → Silver)

**Data Quality Issues to Fix:**
1. Missing values in review_text
2. Duplicate reviews
3. Text normalization (lowercase, whitespace, special chars)
4. Invalid ratings

In [ ]:
print("\n🔧 CLEANING REVIEWS DATASET")
print("=" * 70)

# Start with copy
reviews_clean = reviews_raw.copy()

# Check initial state
print(f"Initial rows: {len(reviews_clean):,}")
print(f"Missing review_text: {reviews_clean['review_text'].isnull().sum()}")
print(f"Duplicates: {reviews_clean.duplicated().sum()}")

# 1. Remove rows with missing review text
before = len(reviews_clean)
reviews_clean = reviews_clean.dropna(subset=['review_text'])
print(f"\n✅ Removed {before - len(reviews_clean)} rows with missing text")

# 2. Remove duplicates
before = len(reviews_clean)
reviews_clean = reviews_clean.drop_duplicates()
print(f"✅ Removed {before - len(reviews_clean)} duplicate rows")

# 3. Clean and normalize text
def clean_text(text):
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = str(text).lower()
    # Remove extra whitespace
    text = ' '.join(text.split())
    # Remove special characters (keep letters, spaces, and basic punctuation)
    text = re.sub(r'[^a-z\s.,!?]', '', text)
    return text

reviews_clean['review_text_clean'] = reviews_clean['review_text'].apply(clean_text)
print(f"✅ Text cleaned and normalized")

# 4. Validate ratings (should be 1-5)
invalid_ratings = reviews_clean[(reviews_clean['rating'] < 1) | (reviews_clean['rating'] > 5)]
print(f"\n⚠️  Invalid ratings found: {len(invalid_ratings)}")
reviews_clean = reviews_clean[(reviews_clean['rating'] >= 1) & (reviews_clean['rating'] <= 5)]

# 5. Add sentiment labels
def rating_to_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating >= 4:
        return 'positive'
    else:
        return 'neutral'

reviews_clean['sentiment'] = reviews_clean['rating'].apply(rating_to_sentiment)
print(f"✅ Sentiment labels added")

# 6. Convert date to datetime
reviews_clean['date'] = pd.to_datetime(reviews_clean['date'])

print(f"\n📊 Final rows: {len(reviews_clean):,}")
print(f"   Removed: {len(reviews_raw) - len(reviews_clean):,} rows ({(len(reviews_raw) - len(reviews_clean))/len(reviews_raw)*100:.1f}%)")

reviews_clean.head()

---

## 🗣️ DISCUSSION POINT: How Should We Handle Missing Values?

**Context**: We found missing values in review_text. What should we do?

### 📊 Current Situation:
- Some reviews have empty/null text
- Rating and product info exist, but no review content

### 🤔 Options:

**A) DROP rows with missing review_text**
```python
reviews_clean = reviews_clean.dropna(subset=['review_text'])
```
- ✅ Pros: Clean data, no garbage in
- ❌ Cons: Lose data, might lose important patterns

**B) FILL with placeholder** (e.g., "No comment")
```python
reviews_clean['review_text'].fillna('No comment provided', inplace=True)
```
- ✅ Pros: Keep all rows, maintain dataset size
- ❌ Cons: Placeholder text might confuse ML model

**C) FILL with neutral sentiment** ("Product was okay")
```python
reviews_clean['review_text'].fillna('Product was okay', inplace=True)
```
- ✅ Pros: Keeps data, neutral assumption
- ❌ Cons: Makes up data that doesn't exist

**D) USE rating to generate text** ("3 star product")
```python
reviews_clean['review_text'].fillna(
    reviews_clean['rating'].apply(lambda x: f'{x} star product')
)
```
- ✅ Pros: Uses available information
- ❌ Cons: Still making up data

---

### 🗳️ VOTE: Which approach should we use?

**Instructor**: Poll the room!
- "Raise hand for A (DROP)"
- "Raise hand for B (placeholder)"
- "Raise hand for C (neutral)"
- "Raise hand for D (use rating)"

**Discussion** (2-3 min):
- Why did you choose that option?
- What are the trade-offs?
- What would you do at your company?

---

### ✅ Our Decision: DROP (Option A)

**Why?**
1. Can't do sentiment analysis without text
2. Making up data is dangerous in production
3. We have enough data (~800 reviews), losing a few is okay
4. **Industry standard**: Don't fabricate missing data

**In production**: You'd investigate WHY reviews are missing (bug? user behavior?) before deciding.